In [81]:
import pandas as pd

file_name = "option-chain-ED-NIFTY-15-Sep-2026.csv"
df= pd.read_csv(file_name, header=1)
df.head()

,Unnamed: 0,OI,CHNG IN OI,VOLUME,IV,LTP,CHNG,BID QTY,BID,ASK,...,BID.1,ASK.1,ASK QTY.1,CHNG.1,LTP.1,IV.1,VOLUME.1,CHNG IN OI.1,OI.1,Unnamed: 22
0,NaN,-,-,-,-,-,-,"1,690","1,795.20","2,198.85",...,3.50,4.55,780,-0.90,4.15,16.44,110,6,250,NaN
1,NaN,-,-,-,-,-,-,"1,690","1,747.35","2,145.45",...,3.90,4.70,325,-1.85,4.15,16.06,33,-,109,NaN
2,NaN,-,-,-,-,-,-,"1,690","1,705.30","2,097.40",...,3.30,17.60,65,-,-,-,-,-,-,NaN
3,NaN,-,-,-,-,-,-,"1,690","1,493.05","2,042.85",...,4.65,5.50,130,-0.85,4.85,15.65,468,95,619,NaN
4,NaN,-,-,-,-,-,-,"1,690","1,447.80","1,994.40",...,4.40,10.75,130,-,-,15.85,-,-,1,NaN


In [82]:
strike_idx = df.columns.get_loc('STRIKE')

df_calls = pd.DataFrame(df.iloc[:, :strike_idx].copy())
df_calls['STRIKE'] = df['STRIKE']
df_calls['SIDE'] = 'Call'

df_puts = pd.DataFrame(df.iloc[:, strike_idx+1:].copy())
df_puts['STRIKE'] = df['STRIKE']
df_puts.columns = [col.replace('.1', '') for col in df_puts.columns]
df_puts['SIDE'] = 'Put'

df_long = pd.concat([df_calls, df_puts], ignore_index=True)

In [83]:
df_long.columns

Index(['Unnamed: 0', 'OI', 'CHNG IN OI', 'VOLUME', 'IV', 'LTP', 'CHNG',
       'BID QTY', 'BID', 'ASK', 'ASK QTY', 'STRIKE', 'SIDE', 'Unnamed: 22'],
      dtype='object')

In [84]:
cols_to_drop = ['Unnamed: 0', 'OI', 'CHNG IN OI', 'CHNG', 'BID QTY', 'ASK QTY', 'Unnamed: 22', 'LTP']
df_long = df_long.drop(cols_to_drop, axis=1)
df_long.head(3)

,VOLUME,IV,LTP,BID,ASK,STRIKE,SIDE
0,-,-,-,"1,795.20","2,198.85","22,350.00",Call
1,-,-,-,"1,747.35","2,145.45","22,400.00",Call
2,-,-,-,"1,705.30","2,097.40","22,450.00",Call


In [89]:
df_long = df_long.rename(columns={'VOLUME': 'volume', 
                                  'IV': 'iv_market',
                                 'BID': 'bid',
                                 'ASK': 'ask',
                                 'STRIKE': 'strike',
                                 'SIDE': 'option_type'})
df_long.columns

Index(['volume', 'iv_market', 'ltp', 'bid', 'ask', 'strike', 'option_type',
       'expiry', 'T'],
      dtype='object')

In [86]:
# Remove the extension and split by dash
parts = file_name.replace(".csv", "").split("-")

expiry = "-".join(parts[4:7])           # '15-Sep-2026'

df_long['expiry'] = expiry
df_long.head(2)

,VOLUME,IV,LTP,BID,ASK,STRIKE,SIDE,expiry
0,-,-,-,"1,795.20","2,198.85","22,350.00",Call,15-Sep-2026
1,-,-,-,"1,747.35","2,145.45","22,400.00",Call,15-Sep-2026


In [87]:
def compute_time_to_expiry(expiry_series, valuation_date='26-Aug-2026'):
    # Convert to datetime and normalize to midnight
    expiry = pd.to_datetime(expiry_series).dt.normalize()
    
    # Convert valuation date to datetime and normalize
    valuation = pd.to_datetime(valuation_date).normalize()
    
    # Calculate days difference (element-wise vectorized)
    delta_days = (expiry - valuation).dt.days
    
    # Convert to years (ACT/365)
    T = delta_days / 365.0
    
    # Safety: Replace T <= 0 with a tiny epsilon (1e-6) to avoid division by zero
    T = T.clip(lower=1e-6)
    
    return T

df_long['T'] = compute_time_to_expiry(df_long['expiry'], valuation_date='26-Aug-2026')
df_long.head(1)

,VOLUME,IV,LTP,BID,ASK,STRIKE,SIDE,expiry,T
0,-,-,-,"1,795.20","2,198.85","22,350.00",Call,15-Sep-2026,0.054795


In [90]:
# 1. Use .str.replace() to remove the commas
# 2. Use pd.to_numeric to convert to float, turning hyphens/errors into NaN
df_long['bid'] = pd.to_numeric(df_long['bid'].astype(str).str.replace(",", ""), errors='coerce')
df_long['ask'] = pd.to_numeric(df_long['ask'].astype(str).str.replace(",", ""), errors='coerce')
df_long['strike'] = pd.to_numeric(df_long['strike'].astype(str).str.replace(",", ""), errors='coerce')

display(df_long.head(1))

,volume,iv_market,ltp,bid,ask,strike,option_type,expiry,T
0,-,-,-,1795.2,2198.85,22350.0,Call,15-Sep-2026,0.054795


In [91]:
df_long['mid_price'] = (df_long['bid'] + df_long['ask']) / 2
df_long.head(1)

,volume,iv_market,ltp,bid,ask,strike,option_type,expiry,T,mid_price
0,-,-,-,1795.2,2198.85,22350.0,Call,15-Sep-2026,0.054795,1997.025


In [92]:
spot = 24281.30
df_long['underlying_price'] = spot
df_long.head(2)

,volume,iv_market,ltp,bid,ask,strike,option_type,expiry,T,mid_price,underlying_price
0,-,-,-,1795.20,2198.85,22350.0,Call,15-Sep-2026,0.054795,1997.025,24281.3
1,-,-,-,1747.35,2145.45,22400.0,Call,15-Sep-2026,0.054795,1946.400,24281.3


In [93]:
risk_free_rate = 0.07
df_long['rate'] = risk_free_rate
df_long.head(2)

,volume,iv_market,ltp,bid,ask,strike,option_type,expiry,T,mid_price,underlying_price,rate
0,-,-,-,1795.20,2198.85,22350.0,Call,15-Sep-2026,0.054795,1997.025,24281.3,0.07
1,-,-,-,1747.35,2145.45,22400.0,Call,15-Sep-2026,0.054795,1946.400,24281.3,0.07


In [104]:
df_long = df_long.replace('-', np.nan)
df_long.isna().sum()

volume              85
iv_market           88
ltp                 85
bid                  1
ask                  8
strike               0
option_type          0
expiry               0
T                    0
mid_price            9
underlying_price     0
rate                 0
dtype: int64

In [106]:
df_long.head(10)

,volume,iv_market,ltp,bid,ask,strike,option_type,expiry,T,mid_price,underlying_price,rate
0,NaN,NaN,NaN,1795.20,2198.85,22350.0,Call,15-Sep-2026,0.054795,1997.025,24281.3,0.07
1,NaN,NaN,NaN,1747.35,2145.45,22400.0,Call,15-Sep-2026,0.054795,1946.400,24281.3,0.07
2,NaN,NaN,NaN,1705.30,2097.40,22450.0,Call,15-Sep-2026,0.054795,1901.350,24281.3,0.07
3,NaN,NaN,NaN,1493.05,2042.85,22500.0,Call,15-Sep-2026,0.054795,1767.950,24281.3,0.07
4,NaN,NaN,NaN,1447.80,1994.40,22550.0,Call,15-Sep-2026,0.054795,1721.100,24281.3,0.07
5,NaN,NaN,NaN,1377.85,1941.25,22600.0,Call,15-Sep-2026,0.054795,1659.550,24281.3,0.07
6,NaN,NaN,NaN,1358.65,1889.40,22650.0,Call,15-Sep-2026,0.054795,1624.025,24281.3,0.07
7,NaN,NaN,NaN,1291.20,1836.55,22700.0,Call,15-Sep-2026,0.054795,1563.875,24281.3,0.07
8,NaN,NaN,NaN,1268.85,1785.40,22750.0,Call,15-Sep-2026,0.054795,1527.125,24281.3,0.07
9,NaN,NaN,NaN,1203.80,1732.35,22800.0,Call,15-Sep-2026,0.054795,1468.075,24281.3,0.07


In [107]:
df_long.tail(10)

,volume,iv_market,ltp,bid,ask,strike,option_type,expiry,T,mid_price,underlying_price,rate
146,NaN,NaN,NaN,1183.10,1725.95,25750.0,Put,15-Sep-2026,0.054795,1454.525,24281.3,0.07
147,NaN,NaN,NaN,1228.85,1774.00,25800.0,Put,15-Sep-2026,0.054795,1501.425,24281.3,0.07
148,NaN,NaN,NaN,1276.00,1830.50,25850.0,Put,15-Sep-2026,0.054795,1553.250,24281.3,0.07
149,NaN,NaN,NaN,1321.80,1793.20,25900.0,Put,15-Sep-2026,0.054795,1557.500,24281.3,0.07
150,NaN,NaN,NaN,1367.85,1848.05,25950.0,Put,15-Sep-2026,0.054795,1607.950,24281.3,0.07
151,1,NaN,"1,555.85",1436.45,1847.55,26000.0,Put,15-Sep-2026,0.054795,1642.000,24281.3,0.07
152,NaN,NaN,NaN,1459.90,1956.00,26050.0,Put,15-Sep-2026,0.054795,1707.950,24281.3,0.07
153,NaN,NaN,NaN,1506.05,2048.20,26100.0,Put,15-Sep-2026,0.054795,1777.125,24281.3,0.07
154,NaN,NaN,NaN,1552.15,2059.95,26150.0,Put,15-Sep-2026,0.054795,1806.050,24281.3,0.07
155,NaN,NaN,NaN,1597.95,2113.80,26200.0,Put,15-Sep-2026,0.054795,1855.875,24281.3,0.07


In [108]:
df_long['volume'] = df_long['volume'].fillna(0)

In [109]:
df_long = df_long[
        (df_long['bid'] > 0) &
        (df_long['ask'] > 0) &
        (df_long['mid_price'] > 0) &
        (df_long['strike'] > 0) &
        (df_long['T'] > 0)
    ].copy()

In [110]:
df_long = df_long[
        (df_long['ask'] / df_long['bid'] < 5.0)
    ].copy()

In [ ]:
def clean_chain(option_chain_path, spot, risk_free_rate):

    # =================================
    # Load Data
    df = pd.read_csv(option_chain_path, header=1)

    # =================================
    # Strip Calls adn Puts separately
    strike_idx = df.columns.get_loc('STRIKE')

    df_calls = pd.DataFrame(df.iloc[:, :strike_idx].copy())                     # calls dataframe
    df_calls['strike'] = df['STRIKE']
    df_calls['option_type'] = 'Call'
    
    df_puts = pd.DataFrame(df.iloc[:, strike_idx+1:].copy())                    # puts dataframe
    df_puts['strike'] = df['STRIKE']
    df_puts.columns = [col.replace('.1', '') for col in df_puts.columns]
    df_puts['option_type'] = 'Put'
    
    df_long = pd.concat([df_calls, df_puts], ignore_index=True)                  # Concatenate the two dataframes


    # =================================
    # Drop Garbage Columns
    cols_to_drop = ['Unnamed: 0', 'OI', 'CHNG IN OI', 'CHNG', 'BID QTY', 'ASK QTY', 'Unnamed: 22', 'LTP']
    df_long = df_long.drop(cols_to_drop, axis=1)

    # =================================
    # Renaming features
    df_long = df_long.rename(columns={'VOLUME': 'volume', 
                                  'IV': 'iv_market',
                                 'BID': 'bid',
                                 'ASK': 'ask'})

    # ==================================
    # Convert to Numeric / Float
    for col in ['bid', 'ask', 'strike']:
        df_long[col] = pd.to_numeric(df_long[col].astype(str).str.replace(",", ""), errors='coerce')


    # ==================================
    # Replacing blank spaces with 0
    df_long = df_long.replace('-', np.nan)
    for col in ['volume', 'market_iv']:
        df_long[col] = df_long[col].fillna(0)
    # ==================================
    # New columns

    # 1. expiry 
    parts = file_name.replace(".csv", "").split("-")                             # Remove the extension and split by dash
    expiry = "-".join(parts[4:7])          
    df_long['expiry'] = expiry

    # 2. T
    def compute_time_to_expiry(expiry_series, valuation_date='26-Aug-2026'):
        expiry = pd.to_datetime(expiry_series).dt.normalize()                    # Convert to datetime and normalize to midnight
        valuation = pd.to_datetime(valuation_date).normalize()                   # Convert valuation date to datetime and normalize
        delta_days = (expiry - valuation).dt.days                                # Calculate days difference (element-wise vectorized)
        T = delta_days / 365.0                                                   # Convert to years (ACT/365)
        T = T.clip(lower=1e-6)                                                   # Safety: Replace T <= 0 with a tiny epsilon (1e-6) to avoid division by zero
        return T
    
    df_long['T'] = compute_time_to_expiry(df_long['expiry'], valuation_date='26-Aug-2026')

    # 3. mid price
    df_long['mid_price'] = (df_long['bid'] + df_long['ask']) / 2

    # 4. underlying spot
    df_long['underlying_price'] = spot

    # 5. rate
    df_long['rate'] = risk_free_rate

    # ==========================================
    # Apply FILTERS
    df_long = df_long[
        (df_long['bid'] > 0) &
        (df_long['ask'] > 0) &
        (df_long['mid_price'] > 0) &
        (df_long['strike'] > 0) &
        (df_long['T'] > 0)
    ].copy()

    # ===========================================
    # Remove strikes with absurdly wide spreads (illiquid garbage)
    df_long = df_long[
        (df_long['ask'] / df_long['bid'] < 5.0)
    ].copy()